In [ ]:
exec(open(__import__('pathlib').Path(__vsc_ipynb_file__).parent.parent / 'src' / 'display_html.py').read())

## Schapiro et al. (2017) — Full Figure Reproduction

This notebook reproduces all three main result figures from:

> Schapiro, A. C., Turk-Browne, N. B., Botvinick, M. M., & Norman, K. A. (2017). Complementary learning systems within the hippocampus. *Phil. Trans. R. Soc. B*, 372, 20160049.

**Testing procedure (§2.d):** present each item in isolation — no previous item, no plus-phase target — and let activity settle for 80 cycles. Record at cycle 20 (initial response) and cycle 80 (settled response).

**Three experiments:**
- **Fig 2** — Pair structure: 8 items in 4 pairs (AB/CD/EF/GH); with vs without statistical learning
- **Fig 3** — Community structure: 15 items in 5 communities × 3 items; MSP/TSP dissociation
- **Fig 4** — Associative inference: 9 items in 3 triads; CA3 recurrence enables transitivity

Note: the paper averages over 500 network seeds. We use `N_REPS` (default 20) for reasonable run time.

In [ ]:
import sys
from pathlib import Path

DIR_SRC = str((Path(__vsc_ipynb_file__).parent.parent / 'src').resolve())
DIR_VIZ = (Path(__vsc_ipynb_file__).parent.parent / 'visualizations').resolve()
DIR_MAN = (Path(__vsc_ipynb_file__).parent.parent / 'manuscript').resolve()
sys.path.insert(0, DIR_SRC)

import numpy as np
import torch
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from model import M_Hip
from tasks import T_PairDataset, T_CommunityGraphDataset
from simulate import run_epoch

# Number of network initializations to average over.
# Schapiro (2017) §2.a.v: 500. We use 20 for practical run time.
N_REPS = 20

print('Setup complete.')

In [ ]:
# =============================================================================
# TESTING PROCEDURE HELPERS
# =============================================================================
# F_run_evaluation / run_evaluation_all are now M_Hip methods (model.py).
# §2.d testing: model.run_evaluation(item_idx) and model.run_evaluation_all()
#
# Remaining helpers here are analysis functions used across all three figures.

def F_pearson_sim_mat(acts_matrix):
    """Pairwise Pearson r similarity matrix.

    Pearson r matches the similarity metric in Schapiro (2017) Figs 2-4.

    Parameters
    ----------
    acts_matrix : ndarray (n_items, n_units)

    Returns
    -------
    ndarray (n_items, n_items), values in [-1, 1]
    """
    centered = acts_matrix - acts_matrix.mean(axis=1, keepdims=True)
    norms    = np.linalg.norm(centered, axis=1, keepdims=True) + 1e-8
    normed   = centered / norms
    return normed @ normed.T


print('Analysis helpers defined.')


## Figure 2 — Pair Structure

Schapiro (2017) §3.a: 8 items in 4 fixed pairs (AB/CD/EF/GH).

**Two conditions** (Fig 2 top and bottom rows):
- **Episodic** (`interleaved=True`): pairs are presented in isolation — A→B, C→D, etc. — with no B→A or cross-pair links. Only TSP can learn (direct pair associations). This is the "without statistical learning" condition.
- **Statistical** (`interleaved=False`): sequential walk — A→B (deterministic), B→A_other (uniform over 3 remaining pairs). Both MSP and TSP learn. MSP detects the forward asymmetry; TSP binds individual A→B episodes.

**Metrics:**
- RSA matrices: 8×8 pairwise Pearson r between CA1 representations (initial and settled)
- Pattern similarity: mean Pearson r for same-pair items vs cross-pair (shuffled baseline)
- Output probability: P(ECout_partner > 0.5) after each epoch — learning curve over 10 epochs

**Expected results (Schapiro 2017 Fig. 2):**
- Both conditions produce CA1 similarity above baseline for same-pair items
- Statistical condition shows smoother community-like RSA structure in CA1 (MSP contribution)
- Episodic condition shows sharper pair-specific patterns (TSP dominates)
- Output probability increases over epochs, faster for episodic (TSP: fast lr=0.4)

In [ ]:
# --- Fig 2: Pair structure training + testing ---
# 8 items (n_items=8), 4 pairs (AB/CD/EF/GH)
# 80 trials/epoch × 10 epochs (Schapiro 2017 §3.a)
# Two conditions × N_REPS seeds each
#
# For output probability curves: test after each epoch (loop, not run_simulation)
# ecout_by_epoch[cond][rep, epoch, item_input, item_output] → float ∈ [0,1]
#   > 0.5 = ECout unit is active = network predicts that item

N_ITEMS_PAIR  = 8
N_EPOCHS_PAIR = 10
N_TRIALS_PAIR = 80    # §3.a
N_PAIRS       = 4

# Pair indices: pair k = items (2k, 2k+1)
PAIRS = [(2*k, 2*k+1) for k in range(N_PAIRS)]

def run_pair_experiment(interleaved, n_reps=N_REPS, seed_base=0):
    """Train N_REPS models on the pair task; test after each epoch.

    Returns
    -------
    settled_by_epoch : ndarray (n_reps, n_epochs, n_items, n_items)
        settled_by_epoch[r, ep, i, j] = ECout activity of unit j when item i is presented,
        after ep+1 epochs of training, for rep r.
    final_rsa : dict[layer] → ndarray (n_reps, n_items, n_items)
        Pearson r RSA matrices from final-epoch settled CA1/CA3/DG representations.
    """
    settled_by_epoch = np.zeros((n_reps, N_EPOCHS_PAIR, N_ITEMS_PAIR, N_ITEMS_PAIR))
    final_sett = {l: [] for l in ['dg', 'ca3', 'ca1']}

    for rep in range(n_reps):
        torch.manual_seed(seed_base + rep)
        np.random.seed(seed_base + rep)
        model = M_Hip(n_items=N_ITEMS_PAIR)

        for ep in range(N_EPOCHS_PAIR):
            dl = T_PairDataset(
                n_steps=N_TRIALS_PAIR, n_pairs=N_PAIRS,
                interleaved=interleaved, seed=seed_base + rep + ep * 1000,
            )
            run_epoch(model, dl, train=True, prev_scale=0.9)

            # Test all 8 items; record settled ECout
            _, sett = model.run_evaluation_all()
            settled_by_epoch[rep, ep] = sett['ecout']   # (n_items, n_items)

        # Final-epoch RSA for DG/CA3/CA1
        for l in ['dg', 'ca3', 'ca1']:
            final_sett[l].append(sett[l])   # sett from last epoch test

    final_rsa = {l: np.array([F_pearson_sim_mat(m) for m in final_sett[l]])
                 for l in ['dg', 'ca3', 'ca1']}
    return settled_by_epoch, final_rsa

print('Running episodic condition...')
ep_ecout, ep_rsa = run_pair_experiment(interleaved=True)
print('Running statistical condition...')
sl_ecout, sl_rsa = run_pair_experiment(interleaved=False)
print('Done.')


In [ ]:
# --- Compute Fig 2 metrics ---
#
# Pattern similarity (Fig 2b/e):
#   For each rep, compute mean Pearson r between settled CA1 reps of paired items.
#   Compare to shuffled baseline: mean r between randomly matched non-paired items.
#
# Output probability (Fig 2c/f):
#   settled_by_epoch[r, ep, A, B] is the ECout activity of unit B when A is input.
#   'Correct' = the designated partner B of item A.
#   Plot mean P(ECout_partner > 0.5) across epochs.

def pair_output_prob(settled, pairs):
    """Mean P(ECout_partner > 0.5) across pairs and reps at each epoch.

    settled : (n_reps, n_epochs, n_items, n_items)
    Returns : (n_epochs,) mean output probability for correct partner
    """
    n_reps, n_epochs = settled.shape[:2]
    correct_probs = np.zeros((n_reps, n_epochs, len(pairs)))
    for r in range(n_reps):
        for ep in range(n_epochs):
            for k, (a, b) in enumerate(pairs):
                # P(ECout_B > 0.5 | input = A)  — forward direction
                correct_probs[r, ep, k] = float(settled[r, ep, a, b] > 0.5)
    return correct_probs.mean(axis=(0, 2))   # mean over reps and pairs


def pair_pattern_sim(rsa_dict, pairs):
    """Mean Pearson r for same-pair vs cross-pair items (across reps).

    Returns
    -------
    same   : dict[layer] → float — mean similarity for paired items
    cross  : dict[layer] → float — mean similarity for non-paired items
    """
    n_items = rsa_dict['ca1'].shape[-1]
    pair_set = set(pairs) | {(b, a) for a, b in pairs}
    same_mask  = np.zeros((n_items, n_items), dtype=bool)
    cross_mask = np.zeros((n_items, n_items), dtype=bool)
    for i in range(n_items):
        for j in range(n_items):
            if i == j:
                continue
            if (i, j) in pair_set:
                same_mask[i, j] = True
            else:
                cross_mask[i, j] = True

    result = {}
    for l, rsas in rsa_dict.items():
        same_vals  = [m[same_mask].mean()  for m in rsas]
        cross_vals = [m[cross_mask].mean() for m in rsas]
        result[l] = {
            'same':  (np.mean(same_vals),  np.std(same_vals)  / np.sqrt(len(same_vals))),
            'cross': (np.mean(cross_vals), np.std(cross_vals) / np.sqrt(len(cross_vals))),
        }
    return result

ep_prob = pair_output_prob(ep_ecout, PAIRS)
sl_prob = pair_output_prob(sl_ecout, PAIRS)
ep_sim  = pair_pattern_sim(ep_rsa, PAIRS)
sl_sim  = pair_pattern_sim(sl_rsa, PAIRS)

# --- Plot Figure 2 ---
fig = plt.figure(figsize=(14, 7))
gs  = gridspec.GridSpec(2, 6, figure=fig, hspace=0.45, wspace=0.5)
epochs = np.arange(1, N_EPOCHS_PAIR + 1)
layers = ['dg', 'ca3', 'ca1']
conditions = [('Episodic (without SL)', ep_rsa, ep_sim, ep_prob),
              ('Statistical (with SL)',  sl_rsa, sl_sim, sl_prob)]

for row, (cond_name, rsa, sim, prob) in enumerate(conditions):
    # --- RSA heatmaps (a/d): DG, CA3, CA1 side by side ---
    for col, layer in enumerate(layers):
        ax = fig.add_subplot(gs[row, col])
        mean_rsa = rsa[layer].mean(axis=0)
        im = ax.imshow(mean_rsa, vmin=-0.5, vmax=1.0, cmap='RdYlBu_r', aspect='auto')
        # Mark pair boundaries
        for k in range(1, N_PAIRS):
            ax.axhline(2*k - 0.5, color='k', lw=0.8)
            ax.axvline(2*k - 0.5, color='k', lw=0.8)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f'{layer.upper()}', fontsize=9)
        if col == 0:
            ax.set_ylabel(cond_name, fontsize=8, rotation=90, labelpad=4)
        if row == 0 and col == 0:
            ax.set_title(f'RSA (settled)\n{layer.upper()}', fontsize=8)

    # --- Pattern similarity bars (b/e) ---
    ax = fig.add_subplot(gs[row, 3])
    x = np.arange(len(layers))
    for i, layer in enumerate(layers):
        same_mean,  same_se  = sim[layer]['same']
        cross_mean, cross_se = sim[layer]['cross']
        ax.bar(i - 0.15, same_mean,  width=0.28, color='C0', alpha=0.8,
               label='same pair' if i == 0 else '')
        ax.bar(i + 0.15, cross_mean, width=0.28, color='C1', alpha=0.8,
               label='cross pair' if i == 0 else '')
        ax.errorbar(i - 0.15, same_mean,  yerr=same_se,  fmt='none', color='k', capsize=3)
        ax.errorbar(i + 0.15, cross_mean, yerr=cross_se, fmt='none', color='k', capsize=3)
    ax.set_xticks(x); ax.set_xticklabels([l.upper() for l in layers], fontsize=8)
    ax.set_ylabel('Pattern similarity (r)', fontsize=8)
    ax.set_ylim(-0.3, 1.1)
    ax.axhline(0, color='k', lw=0.5)
    if row == 0:
        ax.legend(fontsize=7, loc='upper left')
    ax.set_title('Pattern similarity', fontsize=9)

    # --- Output probability curves (c/f) ---
    ax = fig.add_subplot(gs[row, 4:])
    ax.plot(epochs, prob, marker='o', markersize=4, color='C0', label='correct partner')
    ax.axhline(2 / N_ITEMS_PAIR, color='gray', lw=0.8, ls='--', label='chance (k=2/8)')
    ax.set_xlabel('Epoch', fontsize=8)
    ax.set_ylabel('Output probability', fontsize=8)
    ax.set_title('Output probability\n(correct partner)', fontsize=9)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xticks(epochs)
    if row == 0:
        ax.legend(fontsize=7)

fig.suptitle('Figure 2 — Pair Structure (Schapiro 2017)', fontsize=11, fontweight='bold')
plt.savefig(DIR_VIZ / 'Schapiro2017_Fig2_pair.png', dpi=150, bbox_inches='tight')
plt.show()

for cond, sim in [('Episodic', ep_sim), ('Statistical', sl_sim)]:
    print(f'{cond}:')
    for l in layers:
        s = sim[l]['same'][0]; c = sim[l]['cross'][0]
        print(f'  {l.upper():4s}: same-pair={s:.3f}  cross-pair={c:.3f}  diff={s-c:.3f}')

## Figure 3 — Community Structure

Schapiro (2017) §3.b: 15 items in 5 communities × 3 items each.

The community graph has a ring topology: communities form a cycle, and the boundary item of each community connects to the first item of the next community. This produces an asymmetry between **internal** items (connected only within their own community) and **boundary** items (connected to an adjacent community).

**Why this matters:**
- MSP accumulates statistical regularities: items that often co-occur (same community) develop similar CA1 representations. The 15×15 RSA matrix should show 5 warm blocks on the diagonal.
- TSP binds each individual episode: CA3 forms distinct attractors, preserving episode-specific detail.

**Metrics:**
- RSA heatmaps: 15×15 Pearson r (initial vs settled; DG/CA3/CA1)
- Output probability: internal (degree 2) vs boundary (degree 3) items across epochs
  - Boundary items have extra between-community edges → slightly harder to predict correctly
- Settled-minus-initial CA1 heatmap: which items' representations change most through settling?
- Pattern similarity bars: within-community vs between-community (analogous to Fig 2b)

**Expected results (Schapiro 2017 Fig. 3):**
- RSA matrix shows 5 diagonal blocks (community clusters) after training
- Internal items reach slightly higher output probability (more predictable neighbors)
- Settling (20→80 cycles) improves output probability for all items (CA3 pattern completion)

In [ ]:
# --- Fig 3: Community structure training + testing ---
# 15 items, 5 communities × 3 items (Schapiro 2017 Fig. 1)
# 60 trials/epoch × 10 epochs (Schapiro 2017 §3.b)
#
# Community structure:
#   community c: items {3c, 3c+1, 3c+2}
#   Internal items: middle of each community triangle (positions 1, 4, 7, 10, 13)
#     degree = 2 (within-community only)
#   Boundary items: first and last of each community (connected to ring)
#     degree = 3 (2 within + 1 cross-community)

N_ITEMS_COMM = 15
N_COMM       = 5
IPC          = 3    # items per community
N_EPOCHS_COMM = 10
N_TRIALS_COMM = 60  # §3.b

# Item classifications by graph degree
# Internal = middle item of each triangle (positions 1 in 0-indexed community)
INTERNAL_ITEMS  = [c * IPC + 1 for c in range(N_COMM)]       # [1, 4, 7, 10, 13]
BOUNDARY_ITEMS  = [i for i in range(N_ITEMS_COMM) if i not in INTERNAL_ITEMS]


def run_community_experiment(n_reps=N_REPS, seed_base=0):
    """Train N_REPS community graph models; test after each epoch.

    Returns
    -------
    settled_by_epoch : ndarray (n_reps, n_epochs, n_items, n_items) — ECout activity
    initial_rsa, settled_rsa : dict[layer] → ndarray (n_reps, n_items, n_items)
    settled_ca1_all : ndarray (n_reps, n_items, n_CA1) — final-epoch settled CA1
    initial_ca1_all : ndarray (n_reps, n_items, n_CA1) — final-epoch initial CA1
    """
    layers = ['dg', 'ca3', 'ca1']
    n_CA1  = 50  # M_Hip default

    settled_by_epoch = np.zeros((n_reps, N_EPOCHS_COMM, N_ITEMS_COMM, N_ITEMS_COMM))
    init_rsa_lists   = {l: [] for l in layers}
    sett_rsa_lists   = {l: [] for l in layers}
    sett_ca1_list    = []
    init_ca1_list    = []

    for rep in range(n_reps):
        torch.manual_seed(seed_base + rep)
        np.random.seed(seed_base + rep)
        model = M_Hip(n_items=N_ITEMS_COMM)

        # Record initial representations (before any training)
        init_mats, _ = model.run_evaluation_all()
        for l in layers:
            init_rsa_lists[l].append(F_pearson_sim_mat(init_mats[l]))
        init_ca1_list.append(init_mats['ca1'])  # (n_items, n_CA1)

        for ep in range(N_EPOCHS_COMM):
            dl = T_CommunityGraphDataset(
                n_steps=N_TRIALS_COMM,
                n_communities=N_COMM, items_per_community=IPC,
                seed=seed_base + rep + ep * 1000,
            )
            run_epoch(model, dl, train=True, prev_scale=0.9)
            _, sett_mats = model.run_evaluation_all()
            settled_by_epoch[rep, ep] = sett_mats['ecout']

        # Final-epoch settled representations for RSA
        for l in layers:
            sett_rsa_lists[l].append(F_pearson_sim_mat(sett_mats[l]))
        sett_ca1_list.append(sett_mats['ca1'])  # final epoch

    initial_rsa = {l: np.array(init_rsa_lists[l]) for l in layers}
    settled_rsa = {l: np.array(sett_rsa_lists[l]) for l in layers}
    return settled_by_epoch, initial_rsa, settled_rsa, np.array(sett_ca1_list), np.array(init_ca1_list)


print('Running community structure experiment...')
comm_ecout, comm_init_rsa, comm_sett_rsa, comm_sett_ca1, comm_init_ca1 = run_community_experiment()
print('Done.')


In [ ]:
# --- Compute Fig 3 metrics ---
#
# Output probability: for each item i (input), which ECout units activate?
#   P(correct) for internal items vs boundary items — Schapiro 2017 Fig. 3c
#   The paper shows that both item types learn, but internal items (degree 2)
#   achieve slightly higher output probability because their transitions are
#   more predictable (always within-community).
#
# CA1 settled-initial difference:
#   Mean (across reps) of (settled CA1 − initial CA1) for each item.
#   Large positive values = settling strongly reshapes the representation (attractor effect).
#   Schapiro 2017 Fig. 3d: shows which items change most through the settling process.

def comm_output_prob_by_type(settled_ep, items, threshold=0.5):
    """Mean P(ECout_any_neighbor > 0.5) for a subset of items across epochs.

    We consider the output 'correct' if any of the item's community neighbors
    are active (since the target is sampled from them during training).
    """
    # For simplicity: mean max ECout value over all units for each item
    # A more precise measure would need the graph to check neighbors.
    # Here we use mean ECout activity > threshold across all output units
    # as a proxy for "output activated", matching the paper's spirit.
    probs = []
    for ep in range(N_EPOCHS_COMM):
        # settled_ep: (n_reps, n_epochs, n_items, n_items)
        ep_vals = settled_ep[:, ep, :, :]  # (n_reps, n_items, n_items)
        item_probs = []
        for i in items:
            # Max ECout activity when item i is input
            item_probs.append((ep_vals[:, i, :].max(axis=1) > threshold).mean())
        probs.append(np.mean(item_probs))
    return np.array(probs)

comm_prob_internal = comm_output_prob_by_type(comm_ecout, INTERNAL_ITEMS)
comm_prob_boundary = comm_output_prob_by_type(comm_ecout, BOUNDARY_ITEMS)

# CA1 settled-initial difference (mean across reps)
ca1_diff = comm_sett_ca1.mean(axis=0) - comm_init_ca1.mean(axis=0)  # (n_items, n_CA1)
ca1_diff_mat = F_pearson_sim_mat(np.abs(ca1_diff))  # how similar are the change patterns?

# Pattern similarity: within-community vs between-community
community_of = np.array([i // IPC for i in range(N_ITEMS_COMM)])
same_comm_mask  = (community_of[:, None] == community_of[None, :]) & ~np.eye(N_ITEMS_COMM, dtype=bool)
diff_comm_mask  = (community_of[:, None] != community_of[None, :])

within_sim  = [m[same_comm_mask].mean() for m in comm_sett_rsa['ca1']]
between_sim = [m[diff_comm_mask].mean() for m in comm_sett_rsa['ca1']]
print(f'CA1 settled — within-community: {np.mean(within_sim):.3f} ± {np.std(within_sim)/N_REPS**0.5:.3f}')
print(f'CA1 settled — between-community: {np.mean(between_sim):.3f} ± {np.std(between_sim)/N_REPS**0.5:.3f}')

# --- Plot Figure 3 ---
fig = plt.figure(figsize=(16, 8))
gs  = gridspec.GridSpec(2, 6, figure=fig, hspace=0.5, wspace=0.5)
epochs = np.arange(1, N_EPOCHS_COMM + 1)
layers = ['dg', 'ca3', 'ca1']

# Row 0: Initial RSA heatmaps (DG, CA3, CA1)
for col, layer in enumerate(layers):
    ax = fig.add_subplot(gs[0, col])
    mean_rsa = comm_init_rsa[layer].mean(axis=0)
    im = ax.imshow(mean_rsa, vmin=-0.5, vmax=1.0, cmap='RdYlBu_r', aspect='auto')
    for k in range(1, N_COMM):
        ax.axhline(k*IPC - 0.5, color='k', lw=0.8)
        ax.axvline(k*IPC - 0.5, color='k', lw=0.8)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'Initial\n{layer.upper()}', fontsize=9)
    if col == 0:
        ax.set_ylabel('RSA (before training)', fontsize=8)

# Row 1: Settled RSA heatmaps (DG, CA3, CA1)
for col, layer in enumerate(layers):
    ax = fig.add_subplot(gs[1, col])
    mean_rsa = comm_sett_rsa[layer].mean(axis=0)
    im = ax.imshow(mean_rsa, vmin=-0.5, vmax=1.0, cmap='RdYlBu_r', aspect='auto')
    for k in range(1, N_COMM):
        ax.axhline(k*IPC - 0.5, color='k', lw=0.8)
        ax.axvline(k*IPC - 0.5, color='k', lw=0.8)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'Settled\n{layer.upper()}', fontsize=9)
    if col == 0:
        ax.set_ylabel('RSA (after 10 epochs)', fontsize=8)

# Right panels: output probability + pattern similarity
ax = fig.add_subplot(gs[0, 3:5])
ax.plot(epochs, comm_prob_internal, marker='o', markersize=4, color='C0', label='internal (degree 2)')
ax.plot(epochs, comm_prob_boundary, marker='s', markersize=4, color='C1', label='boundary (degree 3)')
ax.axhline(2 / N_ITEMS_COMM, color='gray', lw=0.8, ls='--', label='chance')
ax.set_xlabel('Epoch', fontsize=8); ax.set_ylabel('P(output active)', fontsize=8)
ax.set_title('Output probability\n(internal vs boundary)', fontsize=9)
ax.set_ylim(-0.05, 1.05); ax.set_xticks(epochs[::2])
ax.legend(fontsize=7)

# CA1 settled-initial difference (mean activity change magnitude per item)
ax = fig.add_subplot(gs[0, 5])
diff_mag = np.abs(ca1_diff).mean(axis=1)  # mean magnitude of change per item
colors = ['C0' if i in INTERNAL_ITEMS else 'C1' for i in range(N_ITEMS_COMM)]
ax.bar(range(N_ITEMS_COMM), diff_mag, color=colors, alpha=0.8)
ax.set_xlabel('Item', fontsize=8); ax.set_ylabel('|Δ CA1|', fontsize=8)
ax.set_title('CA1 settled−initial\nmagnitude', fontsize=9)
ax.set_xticks([])

# Pattern similarity: within vs between community
ax = fig.add_subplot(gs[1, 3:])
x = np.arange(len(layers))
for i, layer in enumerate(layers):
    wvals = [m[same_comm_mask].mean() for m in comm_sett_rsa[layer]]
    bvals = [m[diff_comm_mask].mean() for m in comm_sett_rsa[layer]]
    ax.bar(i - 0.15, np.mean(wvals), width=0.28, color='C0', alpha=0.8,
           label='within community' if i == 0 else '',
           yerr=np.std(wvals)/len(wvals)**0.5, capsize=3)
    ax.bar(i + 0.15, np.mean(bvals), width=0.28, color='C1', alpha=0.8,
           label='between community' if i == 0 else '',
           yerr=np.std(bvals)/len(bvals)**0.5, capsize=3)
ax.set_xticks(x); ax.set_xticklabels([l.upper() for l in layers], fontsize=8)
ax.set_ylabel('Pattern similarity (r)', fontsize=8)
ax.set_title('Pattern similarity (settled)', fontsize=9)
ax.axhline(0, color='k', lw=0.5)
ax.legend(fontsize=7)

fig.suptitle('Figure 3 — Community Structure (Schapiro 2017)', fontsize=11, fontweight='bold')
plt.savefig(DIR_VIZ / 'Schapiro2017_Fig3_community.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 4 — Associative Inference

Schapiro (2017) §3.c: 9 items in 3 triads (ABC/DEF/GHI).

The model is trained only on **direct pairs**: A→B and B→C for each triad. The test asks whether it can complete **transitive pairs**: A→C (two hops). This requires inference — the model never sees A and C together during training.

**Why CA3 is needed for transitivity:**
- Without CA3 recurrence: the model stores only direct associations (A→B and B→C). When A is presented at test, ECout can activate B but cannot then use B to activate C.
- With CA3 recurrence: after training, A's CA3 attractor (TSP) encodes A's context including its connection to B. Through recurrent settling, CA3 can bridge the two-hop A→B→C path, producing CA1 patterns that include C-like components.

**Two conditions:**
- **Full model**: normal M_Hip with CA3→CA1 connection
- **No CA3 (lesion)**: CA3→CA1 weights zeroed; only DG→CA1 via MSP (W_ECin→CA1)

**Metrics:**
- RSA heatmaps: 9×9 Pearson r showing whether A and C cluster together (transitive inference)
- Pattern similarity: direct (A↔B, B↔C) vs transitive (A↔C) vs unrelated
- Output probability: P(ECout_C > 0.5 | input=A) for full vs lesion model

In [ ]:
# --- Chain (associative inference) task dataset ---
# 9 items in 3 triads: {0,1,2}, {3,4,5}, {6,7,8}
# Training pairs: (0,1), (1,2), (3,4), (4,5), (6,7), (7,8)
# No connection between triads.
# Schapiro (2017) §3.c: "We trained the model on AB and BC pairs"

N_ITEMS_CHAIN  = 9
N_TRIADS       = 3
IPC_CHAIN      = 3
N_EPOCHS_CHAIN = 10
N_TRIALS_CHAIN = 60

# Direct pairs (trained): A→B, B→C
DIRECT_PAIRS = [(t*IPC_CHAIN + i, t*IPC_CHAIN + i + 1)
                for t in range(N_TRIADS)
                for i in range(IPC_CHAIN - 1)]
# Transitive pairs (NOT trained): A→C (two hops)
TRANS_PAIRS  = [(t*IPC_CHAIN, t*IPC_CHAIN + 2) for t in range(N_TRIADS)]

print('Direct pairs (trained):', DIRECT_PAIRS)
print('Transitive pairs (test):', TRANS_PAIRS)


class T_ChainDataset(Dataset):
    """Chain task: A→B, B→C pairs for associative inference.

    Schapiro (2017) §3.c: trains on direct pairs only.
    Triads are presented in random interleaved order;
    no B→A connections (unlike pair SL task).
    """

    def __init__(self, n_steps=N_TRIALS_CHAIN, n_triads=N_TRIADS, device='cpu', seed=None):
        self.n_items  = n_triads * IPC_CHAIN
        self.n_steps  = n_steps
        self.device   = device
        rng = np.random.default_rng(seed)

        pairs = DIRECT_PAIRS
        idxs  = rng.integers(len(pairs), size=n_steps)
        items      = torch.tensor([pairs[i][0] for i in idxs], dtype=torch.long)
        next_items = torch.tensor([pairs[i][1] for i in idxs], dtype=torch.long)

        n = self.n_items
        self._item_oh   = torch.zeros(n_steps, n)
        self._target_oh = torch.zeros(n_steps, n)
        self._item_oh[torch.arange(n_steps), items]      = 1.0
        self._target_oh[torch.arange(n_steps), next_items] = 1.0

    def __len__(self):
        return self.n_steps

    def __getitem__(self, idx):
        return {
            'item_onehot':   self._item_oh[idx],
            'target_onehot': self._target_oh[idx],
        }


# Quick sanity check
ds = T_ChainDataset(n_steps=12, seed=0)
print(f'Dataset length: {len(ds)}, n_items: {ds.n_items}')
print(f'First batch item: {ds[0]["item_onehot"].argmax().item()}  target: {ds[0]["target_onehot"].argmax().item()}')

In [ ]:
# --- Fig 4: Associative inference training + testing ---
# Two conditions: full model vs CA3 lesion (W_CA3 zeroed)
#
# CA3 lesion: set W_CA3 in L_CA1 to zero before training and freeze it.
# This simulates the "no recurrence" condition in Schapiro 2017 Fig. 4.
# Only MSP (ECin→CA1) and ECout→CA1 back-projection remain.
# DG→CA3 weights still update but CA3→CA1 never contributes to CA1 activity.

def run_chain_experiment(lesion_ca3=False, n_reps=N_REPS, seed_base=0):
    """Train N_REPS chain models; test after all epochs.

    Parameters
    ----------
    lesion_ca3 : bool
        If True, zero out CA3→CA1 weights (and keep them zero throughout training).

    Returns
    -------
    settled_final : ndarray (n_reps, n_items, n_items) — ECout after final epoch
    rsa_final     : dict[layer] → ndarray (n_reps, n_items, n_items)
    """
    layers = ['dg', 'ca3', 'ca1']
    settled_list = []
    rsa_lists    = {l: [] for l in layers}

    for rep in range(n_reps):
        torch.manual_seed(seed_base + rep)
        np.random.seed(seed_base + rep)
        model = M_Hip(n_items=N_ITEMS_CHAIN)

        if lesion_ca3:
            # Zero out CA3→CA1 weights; CA3 activity will compute but never reach CA1
            model.ca1.W_CA3.data.zero_()

        for ep in range(N_EPOCHS_CHAIN):
            dl = T_ChainDataset(
                n_steps=N_TRIALS_CHAIN,
                seed=seed_base + rep + ep * 1000,
            )
            run_epoch(model, dl, train=True, prev_scale=0.9)

            if lesion_ca3:
                # Re-zero after each weight update to maintain the lesion
                model.ca1.W_CA3.data.zero_()

        # Final test
        _, sett_mats = model.run_evaluation_all()
        settled_list.append(sett_mats['ecout'])  # (n_items, n_items)
        for l in layers:
            rsa_lists[l].append(F_pearson_sim_mat(sett_mats[l]))

    settled_final = np.array(settled_list)  # (n_reps, n_items, n_items)
    rsa_final     = {l: np.array(rsa_lists[l]) for l in layers}
    return settled_final, rsa_final


print('Running full model (with CA3 recurrence)...')
chain_full_ecout, chain_full_rsa = run_chain_experiment(lesion_ca3=False)
print('Running lesion model (CA3→CA1 zeroed)...')
chain_les_ecout,  chain_les_rsa  = run_chain_experiment(lesion_ca3=True)
print('Done.')


In [ ]:
# --- Compute Fig 4 metrics ---
#
# Pattern similarity: three categories
#   direct     : A↔B, B↔C (trained pairs within triad)
#   transitive : A↔C (untrained; two-hop)
#   unrelated  : pairs from different triads
#
# Output probability:
#   P(ECout_C > 0.5 | input=A) — transitive output for full vs lesion model
#   P(ECout_B > 0.5 | input=A) — direct output (should be similar in both)

def chain_pair_masks(n_items, n_triads, ipc):
    """Build masks for direct, transitive, and unrelated pairs."""
    direct_set    = set(DIRECT_PAIRS) | {(b,a) for a,b in DIRECT_PAIRS}
    trans_set     = set(TRANS_PAIRS)  | {(b,a) for a,b in TRANS_PAIRS}
    direct_mask   = np.zeros((n_items, n_items), dtype=bool)
    trans_mask    = np.zeros((n_items, n_items), dtype=bool)
    unrelated_mask= np.zeros((n_items, n_items), dtype=bool)
    for i in range(n_items):
        for j in range(n_items):
            if i == j: continue
            if   (i,j) in direct_set:  direct_mask[i,j]    = True
            elif (i,j) in trans_set:   trans_mask[i,j]     = True
            elif i//ipc != j//ipc:     unrelated_mask[i,j] = True
    return direct_mask, trans_mask, unrelated_mask

direct_mask, trans_mask, unrel_mask = chain_pair_masks(N_ITEMS_CHAIN, N_TRIADS, IPC_CHAIN)

def chain_pattern_sim(rsa_dict, masks):
    result = {}
    for l, rsas in rsa_dict.items():
        result[l] = {}
        for name, mask in masks.items():
            vals = [m[mask].mean() for m in rsas]
            result[l][name] = (np.mean(vals), np.std(vals) / len(vals)**0.5)
    return result

masks = {'direct': direct_mask, 'transitive': trans_mask, 'unrelated': unrel_mask}
full_sim = chain_pattern_sim(chain_full_rsa, masks)
les_sim  = chain_pattern_sim(chain_les_rsa,  masks)

# Output probability for transitive (A→C) and direct (A→B) pairs
def chain_output_prob(settled_final, pair_list, threshold=0.5):
    """P(ECout_target > threshold) for given pairs; mean across reps and pairs."""
    probs = []
    for a, b in pair_list:
        probs.append((settled_final[:, a, b] > threshold).mean())
    return np.mean(probs)

full_prob_direct = chain_output_prob(chain_full_ecout, DIRECT_PAIRS)
full_prob_trans  = chain_output_prob(chain_full_ecout, TRANS_PAIRS)
les_prob_direct  = chain_output_prob(chain_les_ecout,  DIRECT_PAIRS)
les_prob_trans   = chain_output_prob(chain_les_ecout,  TRANS_PAIRS)

print('Output probability (direct / transitive):')
print(f'  Full model: direct={full_prob_direct:.3f}  transitive={full_prob_trans:.3f}')
print(f'  Lesion:     direct={les_prob_direct:.3f}  transitive={les_prob_trans:.3f}')

# --- Plot Figure 4 ---
fig = plt.figure(figsize=(16, 6))
gs  = gridspec.GridSpec(1, 8, figure=fig, hspace=0.4, wspace=0.55)
layers = ['dg', 'ca3', 'ca1']
conditions = [('Full model', chain_full_rsa, full_sim),
              ('CA3 lesion (no recurrence)', chain_les_rsa, les_sim)]

for cond_col, (cond_name, rsa, sim) in enumerate(conditions):
    offset = cond_col * 4

    # RSA heatmap for CA1 (most informative layer)
    ax = fig.add_subplot(gs[0, offset])
    mean_rsa = rsa['ca1'].mean(axis=0)
    im = ax.imshow(mean_rsa, vmin=-0.5, vmax=1.0, cmap='RdYlBu_r', aspect='auto')
    for k in range(1, N_TRIADS):
        ax.axhline(k*IPC_CHAIN - 0.5, color='k', lw=0.8)
        ax.axvline(k*IPC_CHAIN - 0.5, color='k', lw=0.8)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'{cond_name}\nCA1 RSA (settled)', fontsize=8)
    for k in range(N_TRIADS):
        ax.text(k*IPC_CHAIN + 1, -0.8, f'T{k}', ha='center', fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # Pattern similarity bars: direct / transitive / unrelated
    ax = fig.add_subplot(gs[0, offset+1:offset+3])
    pair_types = ['direct', 'transitive', 'unrelated']
    colors_bar = ['C0', 'C2', 'C1']
    x = np.arange(len(layers))
    width = 0.22
    for i, layer in enumerate(layers):
        for j, (ptype, col) in enumerate(zip(pair_types, colors_bar)):
            m, se = sim[layer][ptype]
            ax.bar(i + (j-1)*width, m, width=width, color=col, alpha=0.8,
                   label=ptype if i == 0 else '')
            ax.errorbar(i + (j-1)*width, m, yerr=se, fmt='none', color='k', capsize=2)
    ax.set_xticks(x); ax.set_xticklabels([l.upper() for l in layers], fontsize=8)
    ax.set_ylabel('Pattern similarity (r)', fontsize=8)
    ax.set_title('Pattern similarity', fontsize=9)
    ax.axhline(0, color='k', lw=0.5)
    if cond_col == 0:
        ax.legend(fontsize=7)

    # Output probability bar: direct vs transitive
    ax = fig.add_subplot(gs[0, offset+3])
    prob_direct = chain_output_prob(chain_full_ecout if cond_col==0 else chain_les_ecout,
                                     DIRECT_PAIRS)
    prob_trans  = chain_output_prob(chain_full_ecout if cond_col==0 else chain_les_ecout,
                                     TRANS_PAIRS)
    ax.bar([0, 1], [prob_direct, prob_trans], color=['C0', 'C2'], alpha=0.8, width=0.5)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['direct', 'transitive'], fontsize=8)
    ax.set_ylabel('P(output > 0.5)', fontsize=8)
    ax.set_title('Output probability', fontsize=9)
    ax.set_ylim(0, 1.0)
    ax.axhline(2/N_ITEMS_CHAIN, color='gray', lw=0.8, ls='--')

fig.suptitle('Figure 4 — Associative Inference (Schapiro 2017)', fontsize=11, fontweight='bold')
plt.savefig(DIR_VIZ / 'Schapiro2017_Fig4_inference.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nPattern similarity summary (CA1 settled):')
for cond, sim in [('Full', full_sim), ('Lesion', les_sim)]:
    d = sim['ca1']['direct'][0]; t = sim['ca1']['transitive'][0]; u = sim['ca1']['unrelated'][0]
    print(f'  {cond}: direct={d:.3f}  transitive={t:.3f}  unrelated={u:.3f}')